Step 1: Install Required Libraries


In [7]:
# Installing required dependencies
!pip install --upgrade transformers huggingface_hub accelerate

Step 2: Import Required Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

Step 3: Load Pre-trained Transformer Model and Tokenizer

In [2]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


Step 4: Define Chatbot Response Function

In [3]:
# Creating a function for generating chatbot responses
def generate_response(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Converting text into model input tensors
    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    # Generating response tokens from the model
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Extracting only newly generated tokens (excluding input tokens)
    generated_ids = [output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decoding tokens into readable text
    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return response


Step 5: Build Interactive Chatbot Loop

In [4]:
# Starting chatbot interaction loop
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

# Initializing conversation history with system instruction
messages = [
    {"role": "system", "content": "You are a helpful and friendly AI assistant."}
]

while True:
    # Taking user input
    user_input = input("You: ")

    # Checking for exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    # Adding user message to conversation history
    messages.append({
        "role": "user",
        "content": user_input
    })

    # Generating chatbot response based on conversation
    response = generate_response(messages)

    print("Chatbot:", response)
    # Storing chatbot reply for maintaining context
    messages.append({
        "role": "assistant",
        "content": response
    })

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: hi
Chatbot: Hello! How can I assist you today?
You: can u tell me what is data science
Chatbot: Data science is a field of study that involves using statistical, mathematical, and computational methods to analyze large amounts of unstructured or semi-structured data in order to gain insights and make decisions based on patterns and trends within the data. It combines various disciplines such as mathematics, computer science, statistics, programming languages, and business analysis to develop models, algorithms, and techniques for extracting valuable information from complex datasets
You: thank you for your response
Chatbot: You're welcome! If you have any more questions, feel free to ask.
You: exit
Chatbot: Goodbye! Have a great day.
